In [2]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'SimHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../data/train.csv')
feature_cols = [c for c in df.columns if c != 'y']
X = np.nan_to_num(df[feature_cols].values.astype(np.float32))
y = df['y'].values

split = int(len(X) * 0.8)
X_test, y_test = X[split:], y[split:]

anom_indices = np.where(y_test == 1)[0]
breaks = np.where(np.diff(anom_indices) > 1)[0]
seg_starts = np.concatenate([[anom_indices[0]], anom_indices[breaks + 1]])
seg_ends = np.concatenate([anom_indices[breaks], [anom_indices[-1]]])

print(f"测试段: {len(X_test)} 点, 异常: {y_test.sum()} 点, {len(seg_starts)} 段异常")
for i, (s, e) in enumerate(zip(seg_starts, seg_ends)):
    print(f"  段 {i}: [{s}, {e}] 长度={e - s + 1}")

测试段: 27439 点, 异常: 570 点, 19 段异常
  段 0: [14530, 14559] 长度=30
  段 1: [16035, 16064] 长度=30
  段 2: [17137, 17166] 长度=30
  段 3: [17768, 17797] 长度=30
  段 4: [18438, 18467] 长度=30
  段 5: [18900, 18929] 长度=30
  段 6: [19595, 19624] 长度=30
  段 7: [19994, 20023] 长度=30
  段 8: [20400, 20429] 长度=30
  段 9: [21063, 21092] 长度=30
  段 10: [21384, 21413] 长度=30
  段 11: [21601, 21630] 长度=30
  段 12: [22232, 22261] 长度=30
  段 13: [22991, 23020] 长度=30
  段 14: [23842, 23871] 长度=30
  段 15: [24792, 24821] 长度=30
  段 16: [25770, 25799] 长度=30
  段 17: [25812, 25841] 长度=30
  段 18: [26800, 26829] 长度=30


In [ ]:
def plot_features(features, seg_idx, window_size=200):
    n_features = len(features)
    s = seg_starts[seg_idx]
    e = seg_ends[seg_idx]
    
    ws = max(0, s - window_size // 2)
    we = min(len(X_test), e + window_size // 2)
    t = np.arange(ws, we)
    
    h = max(1.2, min(2.5, 60.0 / n_features))
    fig, axes = plt.subplots(n_features, 1, figsize=(14, h * n_features), sharex=True)
    if n_features == 1:
        axes = [axes]
    
    for i, feat in enumerate(features):
        idx = feature_cols.index(feat)
        vals = X_test[ws:we, idx]
        axes[i].plot(t, vals, color='#4a90d9', linewidth=0.6, alpha=0.7)
        
        ai_start = max(ws, s)
        ai_end = min(we, e + 1)
        if ai_start < ai_end:
            axes[i].plot(np.arange(ai_start, ai_end), 
                        X_test[ai_start:ai_end, idx],
                        color='#e74c3c', linewidth=1.2, alpha=0.9)
        
        vmin, vmax = vals.min(), vals.max()
        if vmax > vmin:
            axes[i].fill_betweenx([vmin, vmax], s, e + 1, color='red', alpha=0.08)
        axes[i].axvline(s, color='red', ls='--', lw=0.8, alpha=0.4)
        axes[i].axvline(e + 1, color='red', ls='--', lw=0.8, alpha=0.4)
        
        axes[i].set_ylabel(feat, fontsize=9, fontweight='bold', rotation=0, labelpad=25)
        if i == 0:
            axes[i].set_title(f'异常段 {seg_idx} (索引 [{s}, {e}]) 附近', fontsize=11, fontweight='bold')
        axes[i].grid(alpha=0.15)
    
    axes[-1].set_xlabel('测试段索引', fontsize=10)
    plt.tight_layout(h_pad=0.3)
    plt.show()
    plt.close(fig)

groups = {
    'f1-f10 (均值偏移)': [f'f{i}' for i in range(1, 11)],
    'f11-f15 (动态冻结)': [f'f{i}' for i in range(11, 16)],
    'f16-f21': [f'f{i}' for i in range(16, 22)],
    'f22-f27': [f'f{i}' for i in range(22, 28)],
    'f28-f33': [f'f{i}' for i in range(28, 34)],
    '全部 33 个': feature_cols,
}

@interact(
    group=Dropdown(
        options=list(groups.keys()),
        value='f1-f10 (均值偏移)',
        description='特征组:'
    ),
    seg_idx=IntSlider(
        min=0, max=len(seg_starts)-1, value=0, step=1,
        description='异常段:'
    ),
    window_size=IntSlider(
        min=80, max=500, value=200, step=20,
        description='窗口:'
    ),
)
def interactive_plot(group, seg_idx, window_size):
    plot_features(groups[group], seg_idx, window_size)

interactive(children=(Dropdown(description='特征组:', options=('f1-f10 (均值偏移)', 'f11-f15 (动态冻结)', 'f16-f21', 'f22…

In [ ]:
def plot_features_overlayed(seg_idx, window_size=200):
    """每个特征组叠在同一子图上，共 5 行。"""
    s = seg_starts[seg_idx]
    e = seg_ends[seg_idx]
    
    ws = max(0, s - window_size // 2)
    we = min(len(X_test), e + window_size // 2)
    t = np.arange(ws, we)
    
    overlay_groups = {
        'f1-f5': ['f1', 'f2', 'f3', 'f4', 'f5'],
        'f6-f10': ['f6', 'f7', 'f8', 'f9', 'f10'],
        'f11-f15': ['f11', 'f12', 'f13', 'f14', 'f15'],
        'f16-f21': ['f16', 'f17', 'f18', 'f19', 'f20', 'f21'],
        'f28-f33': ['f28', 'f29', 'f30', 'f31', 'f32', 'f33'],
    }
    
    n_groups = len(overlay_groups)
    fig, axes = plt.subplots(n_groups, 1, figsize=(12, 2.2 * n_groups), sharex=True)
    
    ai = 0
    for gname, feats in overlay_groups.items():
        for feat in feats:
            idx = feature_cols.index(feat)
            vals = X_test[ws:we, idx]
            axes[ai].plot(t, vals, linewidth=0.6, alpha=0.7, label=feat)
            
            ai_start = max(ws, s)
            ai_end = min(we, e + 1)
            if ai_start < ai_end:
                axes[ai].plot(np.arange(ai_start, ai_end),
                              X_test[ai_start:ai_end, idx],
                              linewidth=1.2, alpha=0.9)
        
        vmin = min(X_test[ws:we, feature_cols.index(f)].min() for f in feats)
        vmax = max(X_test[ws:we, feature_cols.index(f)].max() for f in feats)
        if vmax > vmin:
            axes[ai].fill_betweenx([vmin, vmax], s, e + 1, color='red', alpha=0.06)
        axes[ai].axvline(s, color='red', ls='--', lw=0.8, alpha=0.4)
        axes[ai].axvline(e + 1, color='red', ls='--', lw=0.8, alpha=0.4)
        axes[ai].set_ylabel(gname, fontsize=10, fontweight='bold', rotation=0, labelpad=25)
        axes[ai].legend(fontsize=7, loc='upper right', ncol=min(len(feats), 6))
        axes[ai].grid(alpha=0.15)
        ai += 1
    
    axes[-1].set_xlabel('测试段索引', fontsize=10)
    axes[0].set_title(f'异常段 {seg_idx} (索引 [{s}, {e}]) 附近 — 叠图模式', fontsize=11, fontweight='bold')
    plt.tight_layout(h_pad=0.3)
    plt.show()
    plt.close(fig)

@interact(
    seg_idx=IntSlider(
        min=0, max=len(seg_starts)-1, value=0, step=1,
        description='异常段:'
    ),
    window_size=IntSlider(
        min=80, max=500, value=200, step=20,
        description='窗口:'
    ),
)
def interactive_plot_overlayed(seg_idx, window_size):
    plot_features_overlayed(seg_idx, window_size)
